In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Optional, Sequence

import pandas as pd
from tensorboard.backend.event_processing import event_accumulator
import re

In [2]:
def find_tensorboard_event_files(
    root_dir: str | Path,
    pattern: str = "events.out.tfevents.*",
    sort: bool = True,
) -> list[Path]:
    """
    Recursively find TensorBoard event files under a directory.
    """
    root = Path(root_dir).expanduser().resolve()

    if not root.exists():
        raise FileNotFoundError(f"Directory does not exist: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"Expected a directory, got: {root}")

    files = [p for p in root.rglob(pattern) if p.is_file()]

    if sort:
        files = sorted(files)

    return files

In [3]:
def extract_run_metadata(path: str | Path) -> dict:
    """
    Extract metadata from a TensorBoard event file path.

    Expected patterns include paths like:

    1)
    /mnt/asr-data-scaling/outputs/gigaspeech/004/0041/data_scaling/
        data_scaling_frac_0.1_subset_0/runs/events.out.tfevents...

    2)
    /mnt/asr-data-scaling/outputs/gigaspeech/004/0041/
        0041_frac_1.0_subset_0_old/runs/events.out.tfevents...

    Returns
    -------
    dict
        {
            "run_id": "0041",
            "dataset_name": "gigaspeech",
            "fraction": 0.1,
            "type": "old" or None,
        }
    """
    path = Path(path).expanduser().resolve()
    parts = path.parts

    # --- dataset_name ---
    # We expect: ... / outputs / <dataset_name> / ...
    dataset_name = None
    if "outputs" in parts:
        idx = parts.index("outputs")
        if idx + 1 < len(parts):
            dataset_name = parts[idx + 1]

    # --- run_id ---
    # We expect the 4-digit run id after the 3-digit group folder, e.g. /004/0041/
    run_id = None
    for i in range(len(parts) - 1):
        if re.fullmatch(r"\d{3}", parts[i]) and re.fullmatch(r"\d{4}", parts[i + 1]):
            run_id = parts[i + 1]
            break

    # --- fraction and type ---
    # Search across path parts for a token containing frac_<number>
    fraction = None
    type_value = None

    frac_pattern = re.compile(r"frac_([0-9]*\.?[0-9]+)")
    for part in parts:
        frac_match = frac_pattern.search(part)
        if frac_match:
            fraction = float(frac_match.group(1))
            if "old" in part.lower():
                type_value = "old"
            break

    return {
        "run_id": run_id,
        "dataset_name": dataset_name,
        "fraction": fraction,
        "type": type_value,
    }

In [4]:
def load_tensorboard_scalars_to_df(
    event_file: str | Path,
    tags: Optional[Sequence[str]] = None,
    size_guidance: Optional[dict] = None,
) -> pd.DataFrame:
    """
    Load scalar TensorBoard logs from one event file into a DataFrame,
    enriched with metadata extracted from the path.

    Output columns include:
    - run_id
    - dataset_name
    - fraction
    - type
    - tag
    - step
    - value
    - wall_time
    - split
    - event_file
    - run_dir
    """
    event_file = Path(event_file).expanduser().resolve()

    if not event_file.exists():
        raise FileNotFoundError(f"Event file does not exist: {event_file}")
    if not event_file.is_file():
        raise ValueError(f"Expected a file, got: {event_file}")

    if size_guidance is None:
        size_guidance = {
            event_accumulator.SCALARS: 0,
        }

    meta = extract_run_metadata(event_file)

    ea = event_accumulator.EventAccumulator(
        str(event_file),
        size_guidance=size_guidance,
    )
    ea.Reload()

    scalar_tags = ea.Tags().get("scalars", [])
    if not scalar_tags:
        return pd.DataFrame(
            columns=[
                "run_id",
                "dataset_name",
                "fraction",
                "type",
                "tag",
                "step",
                "value",
                "wall_time",
                "split",
                "event_file",
                "run_dir",
            ]
        )

    selected_tags = list(tags) if tags is not None else scalar_tags
    selected_tags = [tag for tag in selected_tags if tag in scalar_tags]

    rows = []
    for tag in selected_tags:
        tag_lower = tag.lower()

        if tag_lower.startswith("train"):
            split = "train"
        elif "val" in tag_lower or "valid" in tag_lower:
            split = "val"
        elif "eval" in tag_lower:
            split = "eval"
        elif "test" in tag_lower:
            split = "test"
        else:
            split = "unknown"

        for ev in ea.Scalars(tag):
            rows.append(
                {
                    "run_id": meta["run_id"],
                    "dataset_name": meta["dataset_name"],
                    "fraction": meta["fraction"],
                    "type": meta["type"],
                    "tag": tag,
                    "step": int(ev.step),
                    "value": float(ev.value),
                    "wall_time": float(ev.wall_time),
                    "split": split,
                    "event_file": str(event_file),
                    "run_dir": str(event_file.parent),
                }
            )

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values(
            ["dataset_name", "run_id", "fraction", "tag", "step"]
        ).reset_index(drop=True)

    return df


def load_all_tensorboard_logs_to_df(
    root_dir: str | Path,
    tags: Optional[Sequence[str]] = None,
    train_only: bool = False,
) -> pd.DataFrame:
    """
    Recursively find all TensorBoard event files and combine them into one DataFrame.

    Adds metadata columns:
    - run_id
    - dataset_name
    - fraction
    - type
    """
    event_files = find_tensorboard_event_files(root_dir)

    all_dfs = []
    for event_file in event_files:
        df = load_tensorboard_scalars_to_df(event_file, tags=tags)

        if df.empty:
            continue

        if train_only:
            df = df[df["split"] == "train"].copy()

        all_dfs.append(df)

    if not all_dfs:
        return pd.DataFrame(
            columns=[
                "run_id",
                "dataset_name",
                "fraction",
                "type",
                "tag",
                "step",
                "value",
                "wall_time",
                "split",
                "event_file",
                "run_dir",
            ]
        )

    combined = pd.concat(all_dfs, ignore_index=True)
    combined = combined.sort_values(
        ["dataset_name", "run_id", "fraction", "tag", "step"]
    ).reset_index(drop=True)

    return combined

In [5]:
root = "/mnt/asr-data-scaling/outputs"

df = load_all_tensorboard_logs_to_df(root, train_only=True)

print(df.head())
print(df.columns.tolist())

/tmp/ipykernel_482553/74503833.py:151: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(all_dfs, ignore_index=True)


  run_id dataset_name  fraction  type          tag  step     value  \
0   0020   gigaspeech       1.0  None  train/epoch    25  0.000588   
1   0020   gigaspeech       1.0  None  train/epoch    25  0.000588   
2   0020   gigaspeech       1.0  None  train/epoch    25  0.000588   
3   0020   gigaspeech       1.0  None  train/epoch    25  0.000588   
4   0020   gigaspeech       1.0  None  train/epoch    25  0.000588   

      wall_time  split                                         event_file  \
0  1.768328e+09  train  /mnt/asr-data-scaling/outputs/gigaspeech/002/0...   
1  1.768296e+09  train  /mnt/asr-data-scaling/outputs/gigaspeech/002/0...   
2  1.768328e+09  train  /mnt/asr-data-scaling/outputs/gigaspeech/002/0...   
3  1.768296e+09  train  /mnt/asr-data-scaling/outputs/gigaspeech/002/0...   
4  1.768244e+09  train  /mnt/asr-data-scaling/outputs/gigaspeech/002/0...   

                                             run_dir  
0  /mnt/asr-data-scaling/outputs/gigaspeech/002/0...  
1  /mn

In [6]:
def get_max_steps_per_fraction(
    df: pd.DataFrame,
    step_col: str = "step",
    dataset_col: str = "dataset_name",
    run_id_col: str = "run_id",
    fraction_col: str = "fraction",
    split_col: str | None = "split",
    split_value: str | None = "train",
) -> pd.DataFrame:
    """
    Aggregate the TensorBoard dataframe to one row per
    (dataset_name, run_id, fraction), holding the maximum step.

    Parameters
    ----------
    df : pd.DataFrame
        Long TensorBoard dataframe.
    step_col, dataset_col, run_id_col, fraction_col : str
        Column names.
    split_col : str | None
        Optional split column, e.g. 'split'.
    split_value : str | None
        If provided, only rows with split == split_value are used.

    Returns
    -------
    pd.DataFrame
        Columns:
        - dataset_name
        - run_id
        - fraction
        - max_step
    """
    work_df = df.copy()

    if split_col is not None and split_value is not None and split_col in work_df.columns:
        work_df = work_df[work_df[split_col] == split_value].copy()

    if work_df.empty:
        return pd.DataFrame(columns=[dataset_col, run_id_col, fraction_col, "max_step"])

    agg_df = (
        work_df.groupby([dataset_col, run_id_col, fraction_col], as_index=False)[step_col]
        .max()
        .rename(columns={step_col: "max_step"})
        .sort_values([dataset_col, run_id_col, fraction_col])
        .reset_index(drop=True)
    )

    return agg_df


def find_run_ids_with_step_mismatch(
    df: pd.DataFrame,
    step_col: str = "step",
    dataset_col: str = "dataset_name",
    run_id_col: str = "run_id",
    fraction_col: str = "fraction",
    split_col: str | None = "split",
    split_value: str | None = "train",
    full_fraction: float = 1.0,
    require_full_fraction: bool = True,
) -> pd.DataFrame:
    """
    Find (dataset_name, run_id) groups where runs with fraction < 1.0
    do not have the same maximum step as the fraction == 1.0 run.

    Rule checked:
        for each dataset_name and run_id,
        max_step(fraction < 1.0) must equal max_step(fraction == 1.0)

    Parameters
    ----------
    df : pd.DataFrame
        Long TensorBoard dataframe.
    step_col, dataset_col, run_id_col, fraction_col : str
        Column names.
    split_col : str | None
        Optional split column to restrict comparison, usually 'train'.
    split_value : str | None
        Value inside split_col to use, usually 'train'.
    full_fraction : float
        The reference full-data fraction, usually 1.0.
    require_full_fraction : bool
        If True, groups without a fraction==1.0 row are flagged as invalid.

    Returns
    -------
    pd.DataFrame
        One row per offending (dataset_name, run_id), with diagnostics.
    """
    max_steps_df = get_max_steps_per_fraction(
        df=df,
        step_col=step_col,
        dataset_col=dataset_col,
        run_id_col=run_id_col,
        fraction_col=fraction_col,
        split_col=split_col,
        split_value=split_value,
    )

    if max_steps_df.empty:
        return pd.DataFrame(
            columns=[
                dataset_col,
                run_id_col,
                "full_fraction_max_step",
                "fractions_below_1",
                "mismatched_fractions",
                "fraction_to_max_step",
                "reason",
            ]
        )

    result_rows = []

    grouped = max_steps_df.groupby([dataset_col, run_id_col], sort=True)

    for (dataset_name, run_id), group in grouped:
        group = group.sort_values(fraction_col).reset_index(drop=True)

        full_rows = group[group[fraction_col] == full_fraction]
        low_rows = group[group[fraction_col] < full_fraction]

        if full_rows.empty:
            if require_full_fraction:
                result_rows.append(
                    {
                        dataset_col: dataset_name,
                        run_id_col: run_id,
                        "full_fraction_max_step": None,
                        "fractions_below_1": sorted(low_rows[fraction_col].tolist()),
                        "mismatched_fractions": sorted(low_rows[fraction_col].tolist()),
                        "fraction_to_max_step": dict(
                            zip(group[fraction_col].tolist(), group["max_step"].tolist())
                        ),
                        "reason": f"missing_fraction_{full_fraction}",
                    }
                )
            continue

        full_max_step = full_rows["max_step"].iloc[0]

        if low_rows.empty:
            # No fractions below 1.0 to compare, so no mismatch
            continue

        mismatched = low_rows[low_rows["max_step"] != full_max_step]

        if not mismatched.empty:
            result_rows.append(
                {
                    dataset_col: dataset_name,
                    run_id_col: run_id,
                    "full_fraction_max_step": full_max_step,
                    "fractions_below_1": sorted(low_rows[fraction_col].tolist()),
                    "mismatched_fractions": sorted(mismatched[fraction_col].tolist()),
                    "fraction_to_max_step": dict(
                        zip(group[fraction_col].tolist(), group["max_step"].tolist())
                    ),
                    "reason": "max_step_mismatch",
                }
            )

    result_df = pd.DataFrame(result_rows)

    if result_df.empty:
        return pd.DataFrame(
            columns=[
                dataset_col,
                run_id_col,
                "full_fraction_max_step",
                "fractions_below_1",
                "mismatched_fractions",
                "fraction_to_max_step",
                "reason",
            ]
        )

    return result_df.sort_values([dataset_col, run_id_col]).reset_index(drop=True)


def get_invalid_run_ids_only(
    df: pd.DataFrame,
    step_col: str = "step",
    dataset_col: str = "dataset_name",
    run_id_col: str = "run_id",
    fraction_col: str = "fraction",
    split_col: str | None = "split",
    split_value: str | None = "train",
) -> list[str]:
    """
    Return unique run_ids that violate the max-step consistency rule.
    """
    invalid_df = find_run_ids_with_step_mismatch(
        df=df,
        step_col=step_col,
        dataset_col=dataset_col,
        run_id_col=run_id_col,
        fraction_col=fraction_col,
        split_col=split_col,
        split_value=split_value,
    )

    if invalid_df.empty:
        return []

    return sorted(invalid_df[run_id_col].astype(str).unique().tolist())

In [7]:
invalid_df = find_run_ids_with_step_mismatch(df)
invalid_df

,dataset_name,run_id,full_fraction_max_step,fractions_below_1,mismatched_fractions,fraction_to_max_step,reason
0,gigaspeech,0023,42504,"[0.1, 0.2, 0.5]","[0.1, 0.2, 0.5]","{0.1: 42490, 0.2: 42490, 0.5: 42490, 1.0: 42504}",max_step_mismatch
1,gigaspeech,0041,42504,"[0.1, 0.2, 0.5]","[0.1, 0.2, 0.5]","{0.1: 42490, 0.2: 42490, 0.5: 32100, 1.0: 42504}",max_step_mismatch
2,gigaspeech,0071,42504,"[0.1, 0.2, 0.5]","[0.1, 0.2, 0.5]","{0.1: 4249, 0.2: 8498, 0.5: 21245, 1.0: 42504}",max_step_mismatch
3,spgispeech_2,0041,9797,"[0.1, 0.2, 0.5]","[0.1, 0.2, 0.5]","{0.1: 979, 0.2: 1959, 0.5: 4898, 1.0: 9797}",max_step_mismatch
4,spgispeech_2,0051,9797,"[0.1, 0.2, 0.5]","[0.1, 0.2, 0.5]","{0.1: 979, 0.2: 1959, 0.5: 4898, 1.0: 9797}",max_step_mismatch
5,spgispeech_2,0061,9797,"[0.1, 0.2, 0.5]","[0.1, 0.2, 0.5]","{0.1: 979, 0.2: 1959, 0.5: 4898, 1.0: 9797}",max_step_mismatch
6,spgispeech_2,0071,9797,"[0.1, 0.2, 0.5]","[0.1, 0.2, 0.5]","{0.1: 979, 0.2: 1959, 0.5: 4898, 1.0: 9797}",max_step_mismatch
7,spgispeech_2,0081,9797,"[0.1, 0.2, 0.5]","[0.1, 0.2, 0.5]","{0.1: 979, 0.2: 1959, 0.5: 4898, 1.0: 9797}",max_step_mismatch


In [13]:
def get_bad_runs_max_steps_long(
    df: pd.DataFrame,
    step_col: str = "step",
    dataset_col: str = "dataset_name",
    run_id_col: str = "run_id",
    fraction_col: str = "fraction",
    split_col: str | None = "split",
    split_value: str | None = "train",
    full_fraction: float = 1.0,
) -> pd.DataFrame:
    """
    Return a long dataframe showing max_step per fraction
    for only the bad runs, including relative percentage
    compared to fraction == 1.0.
    """
    # --- Step 1: compute max steps ---
    max_steps_df = get_max_steps_per_fraction(
        df=df,
        step_col=step_col,
        dataset_col=dataset_col,
        run_id_col=run_id_col,
        fraction_col=fraction_col,
        split_col=split_col,
        split_value=split_value,
    )

    # --- Step 2: get bad runs ---
    bad_runs_df = find_run_ids_with_step_mismatch(
        df=df,
        step_col=step_col,
        dataset_col=dataset_col,
        run_id_col=run_id_col,
        fraction_col=fraction_col,
        split_col=split_col,
        split_value=split_value,
        full_fraction=full_fraction,
    )[[dataset_col, run_id_col, "reason"]].drop_duplicates()

    if bad_runs_df.empty:
        return pd.DataFrame(
            columns=[
                dataset_col,
                run_id_col,
                fraction_col,
                "max_step",
                "full_fraction_max_step",
                "relative_step_ratio",
                "relative_step_percent",
                "reason",
            ]
        )

    # --- Step 3: keep only bad runs ---
    out = max_steps_df.merge(
        bad_runs_df,
        on=[dataset_col, run_id_col],
        how="inner",
    )

    # --- Step 4: compute full-fraction max step per run ---
    full_steps = (
        out[out[fraction_col] == full_fraction]
        [[dataset_col, run_id_col, "max_step"]]
        .rename(columns={"max_step": "full_fraction_max_step"})
    )

    out = out.merge(
        full_steps,
        on=[dataset_col, run_id_col],
        how="left",
    )

    # --- Step 5: compute ratios ---
    out["relative_step_ratio"] = (
        out["max_step"] / out["full_fraction_max_step"]
    )

    out["relative_step_percent"] = (out["relative_step_ratio"] * 100.0).round(0).astype(int)

    # --- Step 6: clean ordering ---
    out = out.sort_values(
        [dataset_col, run_id_col, fraction_col]
    ).reset_index(drop=True)

    return out

In [14]:
bad_max_steps = get_bad_runs_max_steps_long(df)
bad_max_steps


,dataset_name,run_id,fraction,max_step,reason,full_fraction_max_step,relative_step_ratio,relative_step_percent
0,gigaspeech,0023,0.1,42490,max_step_mismatch,42504,0.999671,100
1,gigaspeech,0023,0.2,42490,max_step_mismatch,42504,0.999671,100
2,gigaspeech,0023,0.5,42490,max_step_mismatch,42504,0.999671,100
3,gigaspeech,0023,1.0,42504,max_step_mismatch,42504,1.000000,100
4,gigaspeech,0041,0.1,42490,max_step_mismatch,42504,0.999671,100
5,gigaspeech,0041,0.2,42490,max_step_mismatch,42504,0.999671,100
6,gigaspeech,0041,0.5,32100,max_step_mismatch,42504,0.755223,76
7,gigaspeech,0041,1.0,42504,max_step_mismatch,42504,1.000000,100
8,gigaspeech,0071,0.1,4249,max_step_mismatch,42504,0.099967,10
9,gigaspeech,0071,0.2,8498,max_step_mismatch,42504,0.199934,20
